# Lab Instructions

Create 3 visualizations from text data of your choice.  Each visualization should have at least 1 - 2 sentences explaining both the figure and the interpretation.
You may use any LLM and produce whatever visuals you think best illustrate your data.  

In [ ]:
# 1) Load dataset and compute token frequencies
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from pathlib import Path

# Resolve dataset path in lab or root notebook working directories
base = Path.cwd()
possible_paths = [base / 'Lab' / 'IMDB_Dataset_Cleaned.csv', base / 'IMDB_Dataset_Cleaned.csv']
path = None
for p in possible_paths:
    if p.exists():
        path = p
        break
if path is None:
    raise FileNotFoundError(f"IMDB_Dataset_Cleaned.csv not found in {possible_paths}")

df = pd.read_csv(path)
text_col = 'review' if 'review' in df.columns else df.columns[0]

# Quick cleanup and subset
sample = df.sample(n=min(8000, len(df)), random_state=1)

vectorizer = CountVectorizer(stop_words='english', max_features=1000)
tokens = vectorizer.fit_transform(sample[text_col].astype(str))
word_freq = pd.DataFrame(tokens.sum(axis=0).A1, index=vectorizer.get_feature_names_out(), columns=['count'])
word_freq = word_freq.sort_values('count', ascending=False)

# Plot top 20 words
plt.figure(figsize=(12, 6))
sns.barplot(x=word_freq.head(20).index, y=word_freq.head(20)['count'], palette='viridis')
plt.xticks(rotation=65)
plt.title('Top 20 Most Frequent Words in IMDB Reviews (English stop words removed)')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

### Visualization 1: Most frequent terms
The bar chart above displays the top 20 most frequent words in the IMDB review text after stop-word filtering. Interpretation: common terms like 'movie', 'film', 'like', and 'one' reflect thematic focus around film quality and viewer reactions.

In [ ]:
# 2) Review length distribution, broken down by sentiment label (if available)
if 'sentiment' in sample.columns:
    sample['review_length'] = sample[text_col].astype(str).apply(lambda x: len(x.split()))

    plt.figure(figsize=(10, 6))
    sns.boxplot(x='sentiment', y='review_length', data=sample, palette='Set2')
    plt.title('Review Word Count Distribution by Sentiment')
    plt.xlabel('Sentiment')
    plt.ylabel('Review length (words)')
    plt.tight_layout()
    plt.show()
else:
    print('No sentiment column available for second plot.')

### Visualization 2: Review length by sentiment
The box plot compares review length (word count) for positive vs negative sentiments. Interpretation: longer reviews tend to be positive in this sample, suggesting that positive reviewers may elaborate more, while negative reviews are slightly shorter on average.

In [ ]:
# 3) Sentiment polarity distribution using Vader (or simple heuristic)
try:
    import nltk.sentiment.vader # type: ignore
    import nltk # type: ignore
    nltk.download('vader_lexicon', quiet=True)

    analyzer = nltk.sentiment.vader.SentimentIntensityAnalyzer()
    sample['compound'] = sample[text_col].astype(str).apply(lambda t: analyzer.polarity_scores(t)['compound'])

    plt.figure(figsize=(10, 6))
    sns.histplot(sample['compound'], bins=30, kde=True, color='teal')
    plt.title('Sentiment Polarity (VADER Compound Score) Distribution')
    plt.xlabel('Compound sentiment score')
    plt.ylabel('Review count')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print('VADER sentiment not available:', e)
    print('Fallback: simple polarity by positive and negative keywords')
    pos_words = {'good','great','excellent','amazing','fantastic','best','love','wonderful'}
    neg_words = {'bad','terrible','awful','worst','boring','poor','hate','disappointing'}

    def simple_score(t):
        tks = set(str(t).lower().split())
        return len(pos_words & tks) - len(neg_words & tks)

    sample['compound'] = sample[text_col].astype(str).apply(simple_score)
    plt.figure(figsize=(10, 6))
    sns.histplot(sample['compound'], bins=30, kde=False, color='coral')
    plt.title('Fallback simple keyword sentiment score distribution')
    plt.xlabel('Keyword sentiment score')
    plt.ylabel('Review count')
    plt.tight_layout()
    plt.show()

### Visualization 3: Sentiment score distribution
The histogram visualizes review sentiment polarity across the dataset (using VADER compound scores or a fallback simple keyword approach). Interpretation: The distribution center shows whether sentiment is neutral, positive, or negative overall; in IMDB reviews we usually see a bimodal pattern as reviewers are either strongly positive or strongly negative.